# Data Journalism Project: Mapping and Analyzing Landfills in Serbia
## Investigating Waste Management, Environmental Hazards, and Local Impact

### 1. Introduction and Context
In recent years, waste management has emerged as one of the most critical environmental crises in Serbia. Landfills across the country frequently catch fire—sometimes multiple times a year—releasing toxic smoke, particulate matter, and hazardous pollutants directly into the air. These incidents pose severe health risks and heavily deteriorate the quality of life in nearby local communities. 

The core of the problem lies in systemic infrastructure deficiencies. A vast number of landfills are not managed in accordance with modern environmental and technical standards, and numerous local municipalities lack properly regulated waste disposal systems.

### 2. Objective
This project aims to investigate the current state of waste disposal in the Republic of Serbia. Using official data from the National Open Data Portal (`data.gov.rs`), we will map and analyze these sites to understand their geographic distribution, operational status, and potential risks to surrounding populations.

The dataset categorizes waste disposal sites into three distinct types:
* **Sanitary Landfills (Sanitarne deponije):** Facilities operating in compliance with prescribed sanitary and technical standards.
* **Non-Sanitary Landfills (Nesanitarne deponije):** Official municipal dump sites designated for waste disposal that fail to meet modern technical and environmental regulations.
* **Wild Dumps (Divlje deponije):** Illegal, completely unregulated, and uncontrolled waste disposal sites on public spaces.

### 3. Data Methodology and Stability
To ensure the long-term stability and reproducibility of this data pipeline, this project deliberately relies on downloaded local Excel files (`.xlsx`) rather than making live calls to the portal's API. 

**Why this approach?**
1. **Code Resilience:** Live APIs and government data endpoints are prone to breaking if the portal undergoes structural updates, URL changes, or schema modifications. 
2. **Reproducibility:** Because the source datasets are relatively small in file size, storing them locally within the project repository guarantees that any researcher or peer reviewer can run this Jupyter Notebook and get identical results instantly without depending on external server uptime.

### 4. Data Import and Initial Setup
We will load the three datasets into separate pandas DataFrames. Because the data structure of sanitary landfills differs significantly from non-sanitary and wild dumps, they will be processed through parallel tracks.

In [1]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Define paths to the local data files
DATA_DIR = "./"  # Changes if you move them to a /data subfolder later

sanitary_path = os.path.join(DATA_DIR, "sanitarne_deponije.xlsx")
non_sanitary_path = os.path.join(DATA_DIR, "nesanitarne_deponije.xlsx")
wild_dumps_path = os.path.join(DATA_DIR, "divlje_deponije.xlsx")

# Load the datasets into pandas DataFrames
try:
    df_sanitary = pd.read_excel(sanitary_path)
    df_non_sanitary = pd.read_excel(non_sanitary_path)
    df_wild = pd.read_excel(wild_dumps_path)
    print("All datasets successfully loaded!")
    print(f"-> Sanitary Landfills rows: {len(df_sanitary)}")
    print(f"-> Non-Sanitary Landfills rows: {len(df_non_sanitary)}")
    print(f"-> Wild Dumps rows: {len(df_wild)}")
except Exception as e:
    print(f"Error loading files: {e}")

All datasets successfully loaded!
-> Sanitary Landfills rows: 109
-> Non-Sanitary Landfills rows: 971
-> Wild Dumps rows: 18196


### 5. Architectural Pivot: Three-Chapter Independent Analysis
Upon initial inspection of the raw schemas, it is clear that all three datasets vary significantly in their structure, available variables, and level of detail. Instead of artificially flattening or merging them into a single, generic dataset—which would result in data loss—this project will treat them as three distinct analytical chapters.

Each dataset offers a unique journalistic angle:
1. **Chapter 1: Sanitary Landfills** – A deep dive into modern, regulated facilities, focusing on capacity, operator standards, and official compliance.
2. **Chapter 2: Non-Sanitary Municipal Dumps** – An investigation into official dump sites that fail modern environmental codes, analyzing municipal dependencies and systemic gaps.
3. **Chapter 3: Wild Dumps** – A large-scale geospatial tracking of illegal waste distribution, focusing on proximity to local communities, frequency, and sheer volume.

## Chapter 1: Sanitary Landfills (Sanitarne deponije)
This section isolates and analyzes regulated sanitary facilities in Serbia. Since these are official, managed sites, we will evaluate their distribution, characteristics, and available spatial metadata.

In [3]:
# Previewing Chapter 1 data structure
print(f"Sanitary Landfills shape: {df_sanitary.shape}")

# Display the first 3 rows to see the actual data formatting
df_sanitary.head(3)

Sanitary Landfills shape: (109, 5)


,Санитарна депонија,Географска ширина (latitude),Географска дужина (longitude),Година,Количина отпада (t)
0,РСД „Дубоко“ Ужице,43.871835,19.885259,2015,72051
1,РСД „Дубоко“ Ужице,43.871835,19.885259,2016,77930
2,РСД „Дубоко“ Ужице,43.871835,19.885259,2017,75295


### 5.1 Chapter 1 - Pivoting Long-to-Wide and Year-Over-Year Analysis
To create multi-line charts or small multiples in Datawrapper, the platform requires a wide-format structure where each row is a unique entity (landfill) and each column represents a chronological point in time (years).
To clean our tracking matrix and prevent entity duplication, we will pivot our DataFrame from a "long" format to a "wide" format. This shifts the `year` rows into individual chronological columns. 

This restructuring directly solves our data challenges:
1. It guarantees that each row represents exactly one unique landfill.
2. It natively handles non-sequential records or missing reports (`NaN`) for specific years.
3. It allows us to mathematically isolate years with universal reporting compliance to build a reliable, unskewed aggregate trend chart.

In [5]:
import pandas as pd

# --- STEP 1 & 2: Reshaping the data to Wide Format ---
# We anchor by name and coordinates, and spread the years into columns
df_sanitary_pivot = df_sanitary.pivot_table(
    index=['landfill_name', 'latitude', 'longitude'],
    columns='year',
    values='waste_tonnes'
).reset_index()

# Isolate only the columns that are years (integers)
year_cols = [col for col in df_sanitary_pivot.columns if isinstance(col, int)]

# --- STEP 3: Compliance Check (Identifying complete years) ---
total_landfills = len(df_sanitary_pivot)
print("--- Reporting Status for Datawrapper Timeline ---")
for yr in year_cols:
    reported = df_sanitary_pivot[yr].notna().sum()
    print(f"Year {yr}: {reported}/{total_landfills} landfills reported.")

# --- STEP 4: Exporting the clean dataset for Datawrapper Line Charts / Small Multiples ---
# We keep the name and the year columns for the chart
df_chart_export = df_sanitary_pivot[['landfill_name'] + year_cols]
df_chart_export.to_csv('dw_sanitary_lines_chart.csv', index=False)
print("\n💾 Saved: 'dw_sanitary_lines_chart.csv' (Ready for Datawrapper Line Chart)")

--- Reporting Status for Datawrapper Timeline ---
Year 2015: 9/12 landfills reported.
Year 2016: 10/12 landfills reported.
Year 2017: 10/12 landfills reported.
Year 2018: 10/12 landfills reported.
Year 2019: 11/12 landfills reported.
Year 2020: 11/12 landfills reported.
Year 2021: 12/12 landfills reported.
Year 2022: 12/12 landfills reported.
Year 2023: 12/12 landfills reported.
Year 2024: 12/12 landfills reported.

💾 Saved: 'dw_sanitary_lines_chart.csv' (Ready for Datawrapper Line Chart)


In [6]:
# Select the specific timeframe requested
target_years = [2021, 2022, 2023, 2024]

# Sum up the waste data across all landfills for each target year
national_totals = df_sanitary_pivot[target_years].sum().reset_index()
national_totals.columns = ['year', 'total_waste_tonnes']

# Display the trend table in the notebook
print("--- National Waste Totals (2021-2024) ---")
print(national_totals.to_string(index=False))

# Export to a clean CSV for Datawrapper
national_totals.to_csv('dw_sanitary_national_trend.csv', index=False)
print("\n💾 Saved: 'dw_sanitary_national_trend.csv' (Ready for Datawrapper Trend Chart)")

--- National Waste Totals (2021-2024) ---
 year  total_waste_tonnes
 2021            850115.0
 2022           1294125.0
 2023           1201683.0
 2024           1109461.0

💾 Saved: 'dw_sanitary_national_trend.csv' (Ready for Datawrapper Trend Chart)


### 5.1.1 Adjusting Matrix Orientation for Datawrapper Line Charts (Transposition)
Datawrapper’s line chart and small multiples engine requires a layout where time (chronological years) acts as the primary rows (X-axis), and each individual entity (landfill) serves as a distinct column. We will pivot and transpose our data to match this specific ingestion logic.

In [9]:
# Set landfill names as index, select only the year columns, and transpose (.T) the matrix
df_dw_lines = df_sanitary_pivot.set_index('landfill_name')[year_cols].T

# Save to CSV with 'Year' as the explicit header for the index column
df_dw_lines.to_csv('dw_sanitary_lines_chart.csv', index_label='Year')
print("💾 Successfully exported in Datawrapper-compliant wide format!")

💾 Successfully exported in Datawrapper-compliant wide format!


### 5.2 Compiling the Geospatial Dataset for Scrollytelling (GeoJSON Export)
Unlike off-the-shelf platform maps, a custom scrollytelling web map driven by engines like Mapbox GL JS requires data in standard GeoJSON format. We isolate the latest available reporting metrics for each landfill, convert the coordinates into explicit geospatial geometries, and export the dataset as a spatial layer ready for interactive storytelling and programmatic zoom-triggers.

In [10]:
# --- STEP 5: Isolating Latest Metrics and Creating GeoJSON for Web Mapping ---
def get_latest_metrics(row):
    # Filter out years where this specific landfill actually reported data
    valid_years = [yr for yr in year_cols if pd.notnull(row[yr])]
    if not valid_years:
        return pd.Series([None, None])
    
    # Grab the most recent year and its matching waste value
    latest_yr = max(valid_years)
    return pd.Series([int(latest_yr), row[latest_yr]])

# Apply the locator function row-by-row
df_sanitary_pivot[['latest_reporting_year', 'latest_measured_waste_tonnes']] = df_sanitary_pivot.apply(get_latest_metrics, axis=1)

# Compile into an isolated GeoPandas DataFrame with proper CRS (WGS84)
gdf_sanitary_latest = gpd.GeoDataFrame(
    df_sanitary_pivot[['landfill_name', 'latitude', 'longitude', 'latest_reporting_year', 'latest_measured_waste_tonnes']],
    geometry=[Point(xy) for xy in zip(df_sanitary_pivot['longitude'], df_sanitary_pivot['latitude'])],
    crs="EPSG:4326"
)

# Export directly to GeoJSON for your custom Mapbox website
gdf_sanitary_latest.to_file('sanitary_landfills.geojson', driver='GeoJSON')
print("💾 Saved: 'sanitary_landfills.geojson' (Ready for your Mapbox Scrollytelling site)")

💾 Saved: 'sanitary_landfills.geojson' (Ready for your Mapbox Scrollytelling site)


## Chapter 2 - Initial Ingestion of Non-Sanitary Municipal Dumps
We begin our investigation into Chapter 2 by loading the official registry of non-sanitary municipal dumps (nesanitarne deponije). These sites represent official, municipality-sanctioned dump spots that fail modern environmental and technical codes. We will inspect the available attributes and sample the first 10 rows to understand the data schema.

In [11]:
import pandas as pd

# Load the dataset (adjust the extension .xlsx or .csv if needed)
# If it's a CSV, replace pd.read_excel with pd.read_csv
df_nonsanitary = pd.read_excel('nesanitarne_deponije.xlsx')

print("=== CHAPTER 2: DATASET SHAPE ===")
print(f"Rows: {df_nonsanitary.shape[0]}, Columns: {df_nonsanitary.shape[1]}\n")

print("=== AVAILABLE COLUMNS ===")
for col in df_nonsanitary.columns:
    print(f"- {col}")

print("\n=== FIRST 10 ROWS PREVIEW ===")
display(df_nonsanitary.head(10))

=== CHAPTER 2: DATASET SHAPE ===
Rows: 971, Columns: 49

=== AVAILABLE COLUMNS ===
- Tip deponije
- Godina
- Naziv preduzeća
- PIB
- Matični broj
- Opština
- Mesto
- Šifra mesta
- Poštanski broj
- Ulica i broj
- Telefon
- Telefaks
- Email
- Šifra pretežne delatnosti
- Lokacija (mesto, naselje)
- Naziv
- Geografska širina
- Geografska dužina
- Zauzete katastarske parcele
- Godina početka deponovanja
- Godina završetka deponovanja
- Prosečne godišnje količine otpada, koji se odlaže na nesanitarnu deponiju - smetlište (t)
- Gradovi / opštine ili naselja čiji otpad se odlaže na nesanitarnu deponiju - smetlište
- Ograda oko nesanitarne deponije - smetlišta
- Kapija /rampa na ulazu
- Čuvarska služba
- Kolska vaga
- Drenažni sistem za prikupljanje procednih otpadnih voda
- Sistem za prečišćavanje procednih voda sa semtlišta
- Sistem za otplinavanje deponijskog gasa
- Kojem regionu za upravljanje otpadom pripada vaša opština?
- Sa kojim lokalnim samoupravama imate potpisan sporazum o zajedničk

,Tip deponije,Godina,Naziv preduzeća,PIB,Matični broj,Opština,Mesto,Šifra mesta,Poštanski broj,Ulica i broj,...,Da li se nesanitarna deponija - smetlište ili njen deo nalazi u poplavnom području?,"Da li je za nesanitarno smetlište izrađen Projekat sanacije, zatvaranja i rekultivacije?",Koje godine?,"Da li je pribavljena saglasnost na Projekat sanacije, zatvaranja i rekultivacije?",Koje godine?2,"Da li se izvode radovi po Projektu sanacije, zatvaranja i rekultivacije?","Koji se radovi izvode po Projektu sanacije, zatvaranja i rekultivacije?","Da li se sprovode mere zaštite životne sredine propisane Projektom sanacije, zatvaranja i rekultivacije?","Da li se sprovodi monitoring propisan Projektom sanacije, zatvaranja i rekultivacije?","Da li je potrebna izrada novog ili ažuriranje postojećeg Projekta sanacije, zatvaranja i rekultivacije nesanitarnog smetlišta?"
0,Nesanitarne deponije,2020,"Javno komunalno preduzeće ""Čistoća""",107051272,20732580,Žabalj,"Žabalj, Žabalj, 801321",801321,21230,Svetog Nikole 7,...,Ne,Da,2021.0,Ne,NaN,ne,NaN,Ne,Ne,Ne
1,Nesanitarne deponije,2020,"Javno komunalno preduzeće ""Čistoća""",107051272,20732580,Žabalj,"Žabalj, Žabalj, 801321",801321,21230,Svetog Nikole 7,...,Ne,Ne,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Nesanitarne deponije,2020,"Javno komunalno preduzeće ""Čistoća""",107051272,20732580,Žabalj,"Žabalj, Žabalj, 801321",801321,21230,Svetog Nikole 7,...,Ne,Ne,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Nesanitarne deponije,2020,opština golubac,101483358,7162901,Golubac,"Golubac, Golubac, 712922",712922,12223,Cara lazara 15,...,Ne,Da,2009.0,Da,2009.0,delimicno,Sanacija,Delimično,Ne,Da
4,Nesanitarne deponije,2020,JP KOMUNALAC,100757593,7366612,Trgovište,"Trgovište, Trgovište, 742546",742546,17525,Kralja Petra I Karađorđevića 4,...,Ne,Ne,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Nesanitarne deponije,2020,OPŠTINA KOCELJEVA,101399396,7353561,Koceljeva,"Koceljeva, Koceljeva, 718513",718513,15220,NEMANJINA 74 74,...,Da,Da,2030.0,Ne,NaN,delimicno,Sanacija,Delimično,Delimično,Ne
6,Nesanitarne deponije,2020,OPŠTINA LAJKOVAC,101343119,7353154,Lajkovac,"Lajkovac (varoš), Lajkovac, 723029",723029,14224,OMLADINSKI TRG 1,...,Da,Da,2011.0,Da,2012.0,delimicno,Rekultivacija,Delimično,Delimično,Da
7,Nesanitarne deponije,2020,GRADSKA UPRAVA GRADA ŠAPCA,100084619,7170122,Šabac,"Šabac, Šabac, 746606",746606,15000,GOSPODAR JEVREMOVA 6,...,Da,Da,2019.0,Da,2019.0,ne,NaN,Delimično,Delimično,Ne
8,Nesanitarne deponije,2020,OPŠTINA BAČ,101759575,8012814,Bač,"Bač, Bač, 800236",800236,21420,TRG DR. ZORANA ĐINĐIĆA 2,...,Ne,Ne,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Nesanitarne deponije,2020,OPŠTINSKA UPRAVA MALO CRNIĆE,101336839,7345534,Malo Crniće,"Malo Crniće, Malo Crniće, 727199",727199,12311,Bajlonijeva 119,...,Ne,Da,2012.0,Ne,NaN,delimicno,Sanacija,Ne,Ne,Da


In [13]:
# Group by the exact coordinate columns
coordinate_counts = df_nonsanitary.groupby(['Geografska širina', 'Geografska dužina']).size().reset_index(name='appearance_count')

# Filter to see only repeating coordinates
duplicates = coordinate_counts[coordinate_counts['appearance_count'] > 1].sort_values(by='appearance_count', ascending=False)

print("=== GEOSPATIAL DUPLICATE CHECK ===")
print(f"Total unique geographic locations: {len(coordinate_counts)}")
print(f"Number of locations that repeat: {len(duplicates)}\n")

if len(duplicates) > 0:
    print("Top repeating coordinates and their frequencies:")
    print(duplicates.head(10).to_string(index=False))
else:
    print("✅ Every single row has a 100% unique geographic location. No spatial duplicates found!")

=== GEOSPATIAL DUPLICATE CHECK ===
Total unique geographic locations: 177
Number of locations that repeat: 161

Top repeating coordinates and their frequencies:
 Geografska širina  Geografska dužina  appearance_count
         44.000000          21.000000                20
         43.288472          21.885472                12
         44.729943          21.193231                10
         44.643901          21.179896                10
         45.000000          19.000000                10
         43.599572          20.982063                 8
         44.034167          20.931949                 8
         45.123710          20.171451                 8
         44.745700          21.546400                 8
         43.580309          22.262006                 8


### 6.1.2 Investigative Deep Dive - Understanding Coordinate Repetition
To diagnose the structural reason behind the maximum coordinate replication, we isolate all 20 rows matching the latitude 44.0 and longitude 21.0, displaying their reporting years, administrative regions, and facility names.

In [16]:
# Isolate all 20 rows for the placeholder coordinates
all_placeholders = df_nonsanitary[
    (df_nonsanitary['Geografska širina'] == 44.0) & 
    (df_nonsanitary['Geografska dužina'] == 21.0)
]

# Configure pandas to display all rows without truncation
pd.set_option('display.max_rows', None)

print(f"=== FULL AUDIT: {len(all_placeholders)} ROWS FOR COORDS (44.0, 21.0) ===")
display(all_placeholders[['Godina', 'Opština', 'Mesto', 'Naziv', 'Prosečne godišnje količine otpada, koji se odlaže na nesanitarnu deponiju - smetlište (t)']])

# Reset pandas display option back to default
pd.reset_option('display.max_rows')

=== FULL AUDIT: 20 ROWS FOR COORDS (44.0, 21.0) ===


,Godina,Opština,Mesto,Naziv,"Prosečne godišnje količine otpada, koji se odlaže na nesanitarnu deponiju - smetlište (t)"
37,2020,Petrovac,"Petrovac na Mlavi, Petrovac na Mlavi, 731960","Deponija ""Svine""",8000.0
44,2020,Ćuprija,"Ćuprija, Ćuprija, 744492",mucava,0.0
135,2020,Svilajnac,"Svilajnac, Svilajnac, 738654",Badra,10000.0
192,2023,Petrovac,"Petrovac na Mlavi, Petrovac na Mlavi, 731960","Deponija ""Svine""",8000.0
265,2023,Svilajnac,"Svilajnac, Svilajnac, 738654",Badra,10000.0
368,2018,Ćuprija,"Ćuprija, Ćuprija, 744492",mucava,0.0
385,2018,Petrovac,"Petrovac na Mlavi, Petrovac na Mlavi, 731960","Deponija ""Svine""",10000.0
389,2018,Svilajnac,"Svilajnac, Svilajnac, 738654",Badra,10000.0
417,2017,Topola,"Topola (varošica), Topola, 742228",Torovi,8000.0
431,2017,Ćuprija,"Ćuprija, Ćuprija, 744492",mucava,0.0


In [17]:
# Convert to string and strip any accidental whitespace
df_nonsanitary['Zauzete katastarske parcele_clean'] = df_nonsanitary['Zauzete katastarske parcele'].astype(str).str.strip()

# Group and count frequencies, ignoring rows where data might be missing (like 'nan', 'nepoznato', etc.)
parcel_counts = df_nonsanitary[
    ~df_nonsanitary['Zauzete katastarske parcele_clean'].isin(['nan', 'None', '', '-', 'nepoznato'])
].groupby('Zauzete katastarske parcele_clean').size().reset_index(name='parcel_appearance_count')

# Filter to see repeating cadastral definitions
duplicate_parcels = parcel_counts[parcel_counts['parcel_appearance_count'] > 1].sort_values(by='parcel_appearance_count', ascending=False)

print("=== CADASTRAL PARCEL INTEGRITY CHECK ===")
print(f"Total unique cadastral parcel entries: {len(parcel_counts)}")
print(f"Number of parcel entries that repeat across rows: {len(duplicate_parcels)}\n")

if len(duplicate_parcels) > 0:
    print("Top repeating cadastral parcels and their frequencies:")
    print(duplicate_parcels.head(15).to_string(index=False))
else:
    print("❌ No repeating cadastral entries found. This would suggest erratic reporting formats.")

=== CADASTRAL PARCEL INTEGRITY CHECK ===
Total unique cadastral parcel entries: 167
Number of parcel entries that repeat across rows: 157

Top repeating cadastral parcels and their frequencies:
     Zauzete katastarske parcele_clean  parcel_appearance_count
     18609, 18610 i 18612 KO Požarevac                       10
                      2428 KO Kostolac                       10
              1042/2, 1568/28, 1568/53                        8
                113/1 KAO Novi Sad III                        8
                                 10189                        8
                               4400/11                        8
                         4292 ko Titel                        8
                                  3580                        8
            2539 i 2540 KO Topola Selo                        8
2361,2520,2521,2374,2464,2462KO Halovo                        8
        228/13,228/3,228/4,2344/1,2356                        8
                                  8920

In [18]:
# Isolate all rows matching the exact clean string for the Požarevac parcel
pozaravac_audit = df_nonsanitary[df_nonsanitary['Zauzete katastarske parcele_clean'] == '18609, 18610 i 18612 KO Požarevac']

# Display the core tracking metrics sorted by Year to see the timeline clearly
pozaravac_timeline = pozaravac_audit[[
    'Godina', 
    'Opština', 
    'Naziv', 
    'Geografska širina', 
    'Geografska dužina',
    'Prosečne godišnje količine otpada, koji se odlaže na nesanitarnu deponiju - smetlište (t)'
]].sort_values(by='Godina')

print(f"=== TIMELINE FOR CADASTRAL PARCEL: 18609, 18610 i 18612 KO Požarevac ({len(pozaravac_timeline)} rows) ===")
display(pozaravac_timeline)

=== TIMELINE FOR CADASTRAL PARCEL: 18609, 18610 i 18612 KO Požarevac (10 rows) ===


,Godina,Opština,Naziv,Geografska širina,Geografska dužina,"Prosečne godišnje količine otpada, koji se odlaže na nesanitarnu deponiju - smetlište (t)"
314,2018,Požarevac,Jeremijino polje Požarevac,44.643901,21.179896,35000.000
691,2019,Požarevac,Jeremijino polje Požarevac,44.643901,21.179896,38819.323
110,2020,Požarevac,Jeremijino polje Požarevac,44.643901,21.179896,41080.166
502,2021,Požarevac,Jeremijino polje Požarevac,44.643901,21.179896,35232.826
758,2022,Požarevac,Jeremijino Polje Požarevac,44.643901,21.179896,41873.480
790,2022,Požarevac,Jeremijino polje Požarevac,44.643901,21.179896,41873.479
253,2023,Požarevac,Jeremijino polje Požarevac,44.643901,21.179896,42158.484
202,2023,Požarevac,Jeremijino Polje Požarevac,44.643901,21.179896,42158.480
859,2024,Požarevac,Jeremijino polje Požarevac,44.643901,21.179896,41721.840
862,2024,Požarevac,Jeremijino Polje Požarevac,44.643901,21.179896,41721.840


In [19]:
# Group by the cadastral parcel and count the number of unique years available for each
parcel_year_counts = df_nonsanitary.groupby('Zauzete katastarske parcele_clean')['Godina'].nunique().reset_index(name='year_count')

# Separate into multi-year logs and single-year logs
multi_year_parcels = parcel_year_counts[parcel_year_counts['year_count'] > 1]
single_year_parcels = parcel_year_counts[parcel_year_counts['year_count'] == 1]

print("=== CADASTRAL REPORTING CONTINUITY SUMMARY ===")
print(f"Total unique cadastral entities evaluated: {len(parcel_year_counts)}")
print(f"📈 Sites with MULTI-YEAR reporting history: {len(multi_year_parcels)}")
print(f"📍 Sites with SINGLE-YEAR (one-off) reporting: {len(single_year_parcels)}\n")

# Sort the original dataframe by parcels and years for clean downstream processing
df_nonsanitary_sorted = df_nonsanitary.sort_values(by=['Zauzete katastarske parcele_clean', 'Godina'])

print("=== TOP 10 LONG-TERM REPORTING SITES (BY YEAR COUNT) ===")
display(multi_year_parcels.sort_values(by='year_count', ascending=False).head(10))

=== CADASTRAL REPORTING CONTINUITY SUMMARY ===
Total unique cadastral entities evaluated: 167
📈 Sites with MULTI-YEAR reporting history: 157
📍 Sites with SINGLE-YEAR (one-off) reporting: 10

=== TOP 10 LONG-TERM REPORTING SITES (BY YEAR COUNT) ===


,Zauzete katastarske parcele_clean,year_count
3,10189,8
13,113/1 KAO Novi Sad III,8
8,"1042/2, 1568/28, 1568/53",8
140,8920,8
135,8,8
107,5.68 ha,8
91,4292 ko Titel,8
86,3580,8
84,3415/1,8
93,4400/11,8


In [22]:
# Select the core lifespan columns along with basic identifiers
lifespan_cols = [
    'Godina', 
    'Opština', 
    'Naziv', 
    'Godina početka deponovanja', 
    'Godina završetka deponovanja',
    'Zauzete katastarske parcele_clean'
]

# Extract these columns from our filtered single-year dataset
df_single_year_lifespan = df_single_year_sites[lifespan_cols]

print(f"=== LIFESPAN AUDIT: SINGLE-YEAR REGISTRY ENTRIES ({len(df_single_year_lifespan)} sites) ===")
display(df_single_year_lifespan)

=== LIFESPAN AUDIT: SINGLE-YEAR REGISTRY ENTRIES (10 sites) ===


,Godina,Opština,Naziv,Godina početka deponovanja,Godina završetka deponovanja,Zauzete katastarske parcele_clean
420,2017,Kula,Mesna kancelarija Kruščić,1990,NaN,"1657,1658"
422,2017,Kula,Brdo emušić,1990,NaN,"2785/1,2786"
354,2018,Žitorađa,"""Bare""",1994,2012.0,"1488/1, 1487/1, 1486/1, 1485/1, 1484/1, 8879/13"
348,2018,Žitište,Torak 1,1950,2018.0,100%
352,2018,Vrnjačka Banja,Nesanitarna deponija u selu Gračac,2015,NaN,"218/1, 219/1, 219/4"
411,2018,Plandište,Smetlište komunalnog čvrstog otpada,1980,NaN,"1916, 1917, 1918"
274,2018,Arilje,VIROVO,1965,2004.0,KP Br.1KO Cerova
350,2018,Aleksinac,Lutvina česma,1989,NaN,"KO Aleksinac van varoš: 228/7, 242/4, 247, 248..."
898,2024,Loznica,gradska deponija,1965,NaN,"4528/1, 4529/1, 4529/2, 4530/1, 4530/2, 4530/3..."
946,2024,Beograd-Obrenovac,Grebača,1985,2029.0,163;164;165;161/1;160;154;156;155;138/1;139;140/1


### 6.2 Longitudinal Pivot Analysis: Tracking Structural and Environmental Changes Across Years
To systematically expose reporting shifts, infrastructural upgrades, or stagnant non-compliance, we reshape the dataset into individual multi-year matrices. For each of the 7 core environmental indicators, we track unique landfill entities (isolated by 'Zauzete katastarske parcele') across the available chronological timeline ('Godina'). 

For each indicator, our pipeline programmatically computes:
1. The total number of sites remaining stubbornly non-compliant (**"NE" without changes**).
2. The total number of sites maintaining continuous compliance (**"DA" without changes**).
3. A detailed output of the specific landfills where status transitions occurred (**Dynamic changes over time**).

In [24]:
import pandas as pd
import numpy as np
from collections import Counter

# List of columns to audit
target_columns = [
    "Kapija /rampa na ulazu",
    "Čuvarska služba",
    "Ograda oko nesanitarne deponije - smetlišta",
    "Sistem za otplinavanje deponijskog gasa",
    "Drenažni sistem za prikupljanje procednih otpadnih voda",
    "Sistem za prečišćavanje procednih voda sa semtlišta",
    "Da li je za nesanitarno smetlište izrađen Projekat sanacije, zatvaranja i rekultivacije?"
]

# Quick cleaning
df_clean = df_nonsanitary.copy()
df_clean['Zauzete katastarske parcele_clean'] = df_clean['Zauzete katastarske parcele'].astype(str).str.strip()

for col in target_columns:
    print("\n" + "="*90)
    print(f"📊 TRANSITION AND PEAK YEAR ANALYSIS FOR: {col}")
    print("="*90)
    
    # Filter valid rows
    df_filtered = df_clean[df_clean[col].notnull() & ~df_clean['Zauzete katastarske parcele_clean'].isin(['nan', 'None', '', '-'])]
    if df_filtered.empty:
        print(f"⚠️ No valid data found for column: {col}")
        continue
        
    df_filtered = df_filtered.copy()
    df_filtered[col] = df_filtered[col].astype(str).str.strip().str.lower()
    
    # Create the pivot matrix (Rows = Parcels, Columns = Years)
    matrix = df_filtered.pivot_table(
        index='Zauzete katastarske parcele_clean',
        columns='Godina',
        values=col,
        aggfunc='first'
    )
    
    always_ne_count = 0
    always_da_count = 0
    
    # Dictionaries to track which year the changes happened
    years_ne_to_da = []
    years_da_to_ne = []
    
    total_ne_to_da = 0
    total_da_to_ne = 0
    
    for idx, row in matrix.iterrows():
        # Drop NaNs to look at the continuous reporting sequence for this landfill
        timeline = row.dropna().sort_index()
        unique_responses = set(timeline.tolist())
        
        if not unique_responses:
            continue
            
        # Scenario A: Always NE
        if unique_responses == {'ne'}:
            always_ne_count += 1
        # Scenario B: Always DA
        elif unique_responses == {'da'}:
            always_da_count += 1
        # Scenario C: Changes detected! Let's track the direction and the year
        else:
            # Loop through the reporting years to find the exact transition points
            years_list = timeline.index.tolist()
            for i in range(len(years_list) - 1):
                current_year = years_list[i]
                next_year = years_list[i+1]
                
                current_val = timeline[current_year]
                next_val = timeline[next_year]
                
                # Check for NE -> DA transition
                if current_val == 'ne' and next_val == 'da':
                    total_ne_to_da += 1
                    years_ne_to_da.append(next_year) # The change manifested in the 'next_year'
                    
                # Check for DA -> NE transition
                elif current_val == 'da' and next_val == 'ne':
                    total_da_to_ne += 1
                    years_da_to_ne.append(next_year)

    # Calculate peak years using Counter
    counter_upgrades = Counter(years_ne_to_da)
    counter_downgrades = Counter(years_da_to_ne)
    
    peak_upgrade_year = counter_upgrades.most_common(1)[0] if years_ne_to_da else ("N/A", 0)
    peak_downgrade_year = counter_downgrades.most_common(1)[0] if years_da_to_ne else ("N/A", 0)
    
    # 3. Print final smart analytics
    print(f"📍 Total unique landfills evaluated: {len(matrix)}")
    print(f"🛑 Stubbornly Non-Compliant (ALWAYS 'NE'): {always_ne_count}")
    print(f"✅ Continuously Compliant (ALWAYS 'DA'): {always_da_count}")
    print("-" * 50)
    print(f"📈 Total transitions from 'NE' → 'DA' (Upgrades): {total_ne_to_da}")
    if total_ne_to_da > 0:
        print(f"🔥 PEAK YEAR FOR UPGRADES: {peak_upgrade_year[0]} ({peak_upgrade_year[1]} sites changed to DA)")
        
    print(f"📉 Total transitions from 'DA' → 'NE' (Rollbacks): {total_da_to_ne}")
    if total_da_to_ne > 0:
        print(f"⚠️ PEAK YEAR FOR ROLLBACKS: {peak_downgrade_year[0]} ({peak_downgrade_year[1]} sites ruined/changed to NE)")


📊 TRANSITION AND PEAK YEAR ANALYSIS FOR: Kapija /rampa na ulazu
📍 Total unique landfills evaluated: 167
🛑 Stubbornly Non-Compliant (ALWAYS 'NE'): 45
✅ Continuously Compliant (ALWAYS 'DA'): 85
--------------------------------------------------
📈 Total transitions from 'NE' → 'DA' (Upgrades): 22
🔥 PEAK YEAR FOR UPGRADES: 2023 (6 sites changed to DA)
📉 Total transitions from 'DA' → 'NE' (Rollbacks): 25
⚠️ PEAK YEAR FOR ROLLBACKS: 2020 (7 sites ruined/changed to NE)

📊 TRANSITION AND PEAK YEAR ANALYSIS FOR: Čuvarska služba
📍 Total unique landfills evaluated: 167
🛑 Stubbornly Non-Compliant (ALWAYS 'NE'): 85
✅ Continuously Compliant (ALWAYS 'DA'): 52
--------------------------------------------------
📈 Total transitions from 'NE' → 'DA' (Upgrades): 20
🔥 PEAK YEAR FOR UPGRADES: 2023 (7 sites changed to DA)
📉 Total transitions from 'DA' → 'NE' (Rollbacks): 20
⚠️ PEAK YEAR FOR ROLLBACKS: 2020 (6 sites ruined/changed to NE)

📊 TRANSITION AND PEAK YEAR ANALYSIS FOR: Ograda oko nesanitarne deponi

In [26]:
# List of the three specific infrastructure columns
donut_columns = [
    "Kapija /rampa na ulazu",
    "Čuvarska služba",
    "Ograda oko nesanitarne deponije - smetlišta"
]

# Standardized English mapping for the final output
translation_map = {
    'Oduvek imaju': 'Always Had',
    'Oduvek nemaju': 'Never Had',
    'Nisu imale, pa imaju (Dobile)': 'Gained',
    'Imale, pa nemaju (Izgubile)': 'Lost',
    'Neodređeno / Ostalo': 'Undetermined'
}

# English column names for the final CSV file
column_name_map = {
    "Kapija /rampa na ulazu": "Gate / Ramp",
    "Čuvarska služba": "Guard Service",
    "Ograda oko nesanitarne deponije - smetlišta": "Fencing"
}

# Temporary list to collect data rows for the final export DataFrame
export_rows = []

df_donut_clean = df_nonsanitary.copy()
df_donut_clean['Zauzete katastarske parcele_clean'] = df_donut_clean['Zauzete katastarske parcele'].astype(str).str.strip()

for col in donut_columns:
    # Filter valid records
    df_col = df_donut_clean[df_donut_clean[col].notnull() & ~df_donut_clean['Zauzete katastarske parcele_clean'].isin(['nan', 'None', '', '-'])].copy()
    df_col[col] = df_col[col].astype(str).str.strip().str.lower()
    
    # Extract first and last available year per parcel
    timeline = df_col.sort_values('Godina').groupby('Zauzete katastarske parcele_clean')[col].agg(['first', 'last']).reset_index()
    
    # Internal classification logic
    def classify_trajectory(row):
        start = row['first']
        end = row['last']
        if start == 'da' and end == 'da':
            return 'Oduvek imaju'
        elif start == 'ne' and end == 'ne':
            return 'Oduvek nemaju'
        elif start == 'da' and end == 'ne':
            return 'Imale, pa nemaju (Izgubile)'
        elif start == 'ne' and end == 'da':
            return 'Nisu imale, pa imaju (Dobile)'
        else:
            return 'Neodređeno / Ostalo'

    timeline['Kategorija_SRB'] = timeline.apply(classify_trajectory, axis=1)
    timeline['Status'] = timeline['Kategorija_SRB'].map(translation_map)
    
    # Count the frequencies of each English category
    counts = timeline['Status'].value_counts().to_dict()
    
    # Append the results to our export list
    english_indicator_name = column_name_map[col]
    for status_eng in ['Always Had', 'Never Had', 'Gained', 'Lost']:
        export_rows.append({
            'Indicator': english_indicator_name,
            'Status': status_eng,
            'Count': counts.get(status_eng, 0)
        })

# Create the final flat DataFrame optimized for Datawrapper long-form or split charting
df_dw_export = pd.DataFrame(export_rows)

# Pivot the table so it becomes perfectly scannable: Rows = Status, Columns = Indicators
df_dw_pivot = df_dw_export.pivot(index='Status', columns='Indicator', values='Count').reset_index()

# Save to CSV for direct upload to Datawrapper
output_file_path = "donut_charts_data.csv"
df_dw_pivot.to_csv(output_file_path, index=False)

print("=== DATAWRAPPER EXPORT SUCCESSFUL ===")
print(f"File saved as: '{output_file_path}' in your current working directory.")
print("\n PREVIEW OF THE EXPORTED DATA (READY FOR DATAWRAPPER):")
display(df_dw_pivot)

=== DATAWRAPPER EXPORT SUCCESSFUL ===
File saved as: 'donut_charts_data.csv' in your current working directory.

 PREVIEW OF THE EXPORTED DATA (READY FOR DATAWRAPPER):


Indicator,Status,Fencing,Gate / Ramp,Guard Service
0,Always Had,80,93,57
1,Gained,14,12,11
2,Lost,9,15,11
3,Never Had,64,47,88


### 6.3 Extracting Latest Profiles and Geospatial Coordinates for Mapbox Integration
To prepare our primary spatial layer for Mapbox and QGIS, we isolate the most recent reporting year for each unique physical landfill based on its cadastral parcel. For this final snapshot, we evaluate the current status of the three core security features ('Fencing', 'Gate', 'Guard'), mapping them strictly to binary values ('da' / 'ne'). 

Geospatial coordinates (Latitude and Longitude) are preserved as-is, allowing for visual inspection, deduplication, and manual geometry corrections directly within QGIS.

In [36]:
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

# 1. Clean the cadastral parcel column just in case (removing trailing/leading spaces)
df_nonsanitary['Zauzete katastarske parcele_clean'] = df_nonsanitary['Zauzete katastarske parcele'].astype(str).str.strip()

# 2. Sort chronologically by parcel and year so the latest report is at the bottom
df_map_sorted = df_nonsanitary.sort_values(by=['Zauzete katastarske parcele_clean', 'Godina'])

# 3. Drop duplicates keeping strictly the LAST row for each unique cadastral parcel
df_latest_map_profiles = df_map_sorted.drop_duplicates(subset=['Zauzete katastarske parcele_clean'], keep='last').copy()

# Helper function to strictly map to 'da' or 'ne'
def normalize_binary(val):
    val_str = str(val).strip().lower()
    return 'da' if val_str == 'da' else 'ne'

# 4. Process security indicators
df_latest_map_profiles['fencing'] = df_latest_map_profiles['Ograda oko nesanitarne deponije - smetlišta'].apply(normalize_binary)
df_latest_map_profiles['gate'] = df_latest_map_profiles['Kapija /rampa na ulazu'].apply(normalize_binary)
df_latest_map_profiles['guard'] = df_latest_map_profiles['Čuvarska služba'].apply(normalize_binary)

# 5. Map and rename final target columns
mapbox_columns = {
    'Naziv': 'landfill_name',
    'Opština': 'municipality',
    'Zauzete katastarske parcele_clean': 'cadastral_parcel',
    'Godina': 'latest_reporting_year',
    'fencing': 'fencing',
    'gate': 'gate',
    'guard': 'guard',
    'Geografska širina': 'latitude', 
    'Geografska dužina': 'longitude'
}

df_mapbox_final = df_latest_map_profiles[list(mapbox_columns.keys())].rename(columns=mapbox_columns)

# 6. Ensure coordinate columns are strictly numeric
df_mapbox_final['latitude'] = pd.to_numeric(df_mapbox_final['latitude'], errors='coerce')
df_mapbox_final['longitude'] = pd.to_numeric(df_mapbox_final['longitude'], errors='coerce')

# Drop rows where coordinates are completely missing or broken before spatial translation
df_spatial_clean = df_mapbox_final.dropna(subset=['latitude', 'longitude']).copy()

# 7. Convert to GeoDataFrame (Longitude always comes FIRST in GeoJSON point generation)
geometry = [Point(xy) for xy in zip(df_spatial_clean['longitude'], df_spatial_clean['latitude'])]
gdf_mapbox = gpd.GeoDataFrame(df_spatial_clean, geometry=geometry, crs="EPSG:4326")

# Note: Raw 'latitude' and 'longitude' columns are purposefully kept in the dataframe 
# so they are fully accessible inside the QGIS Attribute Table.

# 8. Export to formal GeoJSON
map_output_geojson = "mapbox_landfills_latest.geojson"
gdf_mapbox.to_file(map_output_geojson, driver='GeoJSON')

print("=== MAPBOX/QGIS CADASTRAL-ANCHORED EXPORT SUCCESSFUL ===")
print(f"Total unique cadastral entities exported to map: {gdf_mapbox.shape[0]}")
print(f"File updated and saved as: '{map_output_geojson}'")
print("\n📋 PREVIEW OF THE SPATIAL DATA:")
display(gdf_mapbox.head(5))

=== MAPBOX/QGIS CADASTRAL-ANCHORED EXPORT SUCCESSFUL ===
Total unique cadastral entities exported to map: 167
File updated and saved as: 'mapbox_landfills_latest.geojson'

📋 PREVIEW OF THE SPATIAL DATA:


,landfill_name,municipality,cadastral_parcel,latest_reporting_year,fencing,gate,guard,latitude,longitude,geometry
868,Podrimce,Leskovac,"1,2,223/1 KO Podrimce",2024,ne,ne,ne,43.140900,21.512600,POINT (21.5126 43.1409)
955,Gradska deponija,Zrenjanin,100,2024,ne,da,da,45.354047,20.364461,POINT (20.36446 45.35405)
348,Torak 1,Žitište,100%,2018,da,da,ne,45.519715,20.596107,POINT (20.59611 45.51971)
950,Obla,Boljevac,10189,2024,da,ne,ne,43.807785,21.872181,POINT (21.87218 43.80778)
870,Gradska deponija,Negotin,"10251,10258",2024,da,da,ne,44.239864,22.556559,POINT (22.55656 44.23986)


### 6.4.2 Harmonizing Historical Data Using Refined Geospatial Coordinates
Following physical verification and geometric deduplication within QGIS, the finalized spatial layer (`clean_aftermap`) revealed that several historical records shared identical coordinate pairs, exposing structural shifts in nomenclature and parcel identifiers over time. This block imports the cleaned spatial attributes and executes a programmatic update back into the master dataframe (`df_nonsanitary`). By mapping the refined latitude and longitude points onto the historical timeline using a dual-key configuration (Municipality and Landfill Name), we preserve the complete longitudinal reporting history while establishing a definitive, lower count of actual physical waste sites across Serbia.

In [38]:
import pandas as pd

# Load the cleaned QGIS data
clean_aftermap = pd.read_csv("clean_aftermap.csv")

# Create clean, lowercase join keys to ensure perfect matching
for df, name_col, muni_col in [(df_nonsanitary, 'Naziv', 'Opština'), (clean_aftermap, 'landfill_name', 'municipality')]:
    df['name_key'] = df[name_col].astype(str).str.strip().str.lower()
    df['muni_key'] = df[muni_col].astype(str).str.strip().str.lower()

# Isolate verified coordinates and update the master historical dataframe
coords = clean_aftermap[['name_key', 'muni_key', 'latitude', 'longitude']].drop_duplicates()
df_nonsanitary = df_nonsanitary.drop(columns=['Geografska širina', 'Geografska dužina']).merge(
    coords, on=['name_key', 'muni_key'], how='left'
).rename(columns={'latitude': 'Geografska širina', 'longitude': 'Geografska dužina'}).drop(columns=['name_key', 'muni_key'])

print("=== HISTORICAL DATABASE HARMONIZATION SUCCESSFUL ===\n")

# Calculate and display the final refined metrics
final_spatial_counts = df_nonsanitary.groupby(['Geografska dužina', 'Geografska širina'])['Godina'].nunique().reset_index(name='year_count')

print("=== FINAL REFINED GEOSPATIAL SUMMARY ===")
print(f"Actual unique physical landfill sites verified on the ground: {len(final_spatial_counts)}")
print(f"📈 Verified sites with MULTI-YEAR reporting history: {len(final_spatial_counts[final_spatial_counts['year_count'] > 1])}")
print(f"📍 Verified sites with SINGLE-YEAR (one-off) reporting: {len(final_spatial_counts[final_spatial_counts['year_count'] == 1])}")

=== HISTORICAL DATABASE HARMONIZATION SUCCESSFUL ===

=== FINAL REFINED GEOSPATIAL SUMMARY ===
Actual unique physical landfill sites verified on the ground: 154
📈 Verified sites with MULTI-YEAR reporting history: 150
📍 Verified sites with SINGLE-YEAR (one-off) reporting: 4


In [39]:
# Save the harmonized master database to a new file
df_nonsanitary.to_csv("df_nonsanitary_clean_master.csv", index=False)